# UTS Data Acquisition & Management
## Sebaran Daya Tarik Wisata di Provinsi Bali


Identitas Mahasiswa

Nama: I Kadek Ananta Wijaya Astawa

NIM: 2501010124

Program Studi: Informatika MDA

Mata Kuliah: Data Acquisition & Management

Dosen: Dr. I Gede Totok Suryawan, S.Kom., M.T.

In [1]:
!pip -q install pandas requests beautifulsoup4 lxml openpyxl

In [2]:
from pathlib import Path
from datetime import datetime, timezone
import requests, pandas as pd, json, hashlib, time, re, unicodedata, os
from bs4 import BeautifulSoup
from urllib.robotparser import RobotFileParser
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
from IPython.display import display

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
RAW = ROOT / "data" / "raw"
PROCESSED = ROOT / "data" / "processed"
DOCS = ROOT / "docs"
for folder in [RAW, PROCESSED, DOCS]:
    folder.mkdir(parents=True, exist_ok=True)

EMAIL_KAMPUS = "ikadekanantawijayaastawa@gmail.com"
if "GANTI_DENGAN" in EMAIL_KAMPUS or "@" not in EMAIL_KAMPUS:
    raise ValueError("Ganti EMAIL_KAMPUS dengan email kampus pribadi Anda.")

USER_AGENT = f"UTS-DAM-Wisata-JawaBarat/1.0 ({EMAIL_KAMPUS})"
TIMEOUT = 45
DELAY = 2

session = requests.Session()
session.headers.update({"User-Agent": USER_AGENT})
retry = Retry(total=3, connect=3, read=3, backoff_factor=1,
              status_forcelist=[429, 500, 502, 503, 504],
              allowed_methods=["GET", "POST"])
adapter = HTTPAdapter(max_retries=retry)
session.mount("https://", adapter)
session.mount("http://", adapter)

def request_checked(method, url, **kwargs):
    response = session.request(method, url, timeout=TIMEOUT, **kwargs)
    print("Request:", response.url, "| Status:", response.status_code)
    response.raise_for_status()
    time.sleep(DELAY)
    return response

def md5_file(path):
    h = hashlib.md5()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024*1024), b""):
            h.update(chunk)
    return h.hexdigest()

def normalize_name(value):
    value = unicodedata.normalize("NFKD", str(value))
    value = "".join(ch for ch in value if not unicodedata.combining(ch))
    value = value.lower().strip()
    value = re.sub(r"^(pantai|pura|danau|air terjun|desa wisata|museum)\s+", "", value)
    value = re.sub(r"[^a-z0-9]+", " ", value)
    return re.sub(r"\s+", " ", value).strip()

print("Root:", ROOT)
print("User-Agent:", USER_AGENT)

Root: /content
User-Agent: UTS-DAM-Wisata-JawaBarat/1.0 (ikadekanantawijayaastawa@gmail.com)


In [3]:
import pandas as pd
from pathlib import Path
import hashlib

# Lokasi folder project
PROJECT_DIR = Path.cwd()

# Lokasi dataset lokal
RAW = PROJECT_DIR / "data" / "raw"

# Nama file dataset (disesuaikan dengan file yang tersedia)
raw_dataset_path = RAW / "banyaknya__pengunjung_daya_tarik_wisata_per_bulan.csv"

# Memastikan folder dan file tersedia
if not RAW.exists():
    raise FileNotFoundError(
        f"Folder dataset tidak ditemukan: {RAW}"
    )

if not raw_dataset_path.exists():
    raise FileNotFoundError(
        f"File dataset tidak ditemukan: {raw_dataset_path}"
    )

# Membaca dataset lokal
df = pd.read_csv(raw_dataset_path)

# Menghitung MD5
def md5_file(path):
    md5 = hashlib.md5()

    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(4096), b""):
            md5.update(chunk)

    return md5.hexdigest()

# Menampilkan informasi dataset
print("Dataset berhasil dibaca dari penyimpanan lokal")
print("Nama file:", raw_dataset_path.name)
print("Lokasi file:", raw_dataset_path)
print("Ukuran:", raw_dataset_path.stat().st_size, "bytes")
print("MD5:", md5_file(raw_dataset_path))

print("\nJumlah baris dan kolom:", df.shape)
print("\nInformasi kolom:")
display(df.info())

print("\nPreview dataset:")
display(df.head())

Dataset berhasil dibaca dari penyimpanan lokal
Nama file: banyaknya__pengunjung_daya_tarik_wisata_per_bulan.csv
Lokasi file: /content/data/raw/banyaknya__pengunjung_daya_tarik_wisata_per_bulan.csv
Ukuran: 12580 bytes
MD5: a1102ac62be2d85ce53d31ca0e39fc51

Jumlah baris dan kolom: (168, 10)

Informasi kolom:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 168 entries, 0 to 167
Data columns (total 10 columns):
 #   Column                   Non-Null Count  Dtype 
---  ------                   --------------  ----- 
 0   id                       168 non-null    int64 
 1   kode_provinsi            168 non-null    int64 
 2   nama_provinsi            168 non-null    object
 3   bps_kode_kabupaten_kota  168 non-null    int64 
 4   bps_nama_kabupaten_kota  168 non-null    object
 5   bulan                    168 non-null    object
 6   asal_wisatawan           168 non-null    object
 7   jumlah_wisatawan         168 non-null    int64 
 8   satuan                   168 non-null    object
 9  

None


Preview dataset:


,id,kode_provinsi,nama_provinsi,bps_kode_kabupaten_kota,bps_nama_kabupaten_kota,bulan,asal_wisatawan,jumlah_wisatawan,satuan,tahun
0,1,32,JAWA BARAT,3208,KABUPATEN\n KUNINGAN,JANUARI,WISNUS,282584,ORANG,2016
1,2,32,JAWA BARAT,3208,KABUPATEN KUNINGAN,JANUARI,WISMAN,9,ORANG,2016
2,3,32,JAWA BARAT,3208,KABUPATEN\n KUNINGAN,JANUARI,WISNUS,165558,ORANG,2017
3,4,32,JAWA BARAT,3208,KABUPATEN KUNINGAN,JANUARI,WISMAN,9,ORANG,2017
4,5,32,JAWA BARAT,3208,KABUPATEN\n KUNINGAN,JANUARI,WISNUS,161808,ORANG,2018


In [18]:
if raw_dataset_path.suffix == ".csv":
    try:
        df_dataset_raw = pd.read_csv(raw_dataset_path, sep=None, engine="python", encoding="utf-8-sig")
    except UnicodeDecodeError:
        df_dataset_raw = pd.read_csv(raw_dataset_path, sep=None, engine="python", encoding="latin-1")
elif raw_dataset_path.suffix in [".xlsx", ".xls"]:
    df_dataset_raw = pd.read_excel(raw_dataset_path, sheet_name=0)
elif raw_dataset_path.suffix == ".json":
    df_dataset_raw = pd.read_json(raw_dataset_path)
else:
    raise ValueError(f"Format file belum didukung: {raw_dataset_path.suffix}")

df_dataset_raw.columns = [str(c).strip() for c in df_dataset_raw.columns]
print("Jenis data: terstruktur")
print("Ukuran dataset:", df_dataset_raw.shape)
print("Ukuran file:", raw_dataset_path.stat().st_size, "bytes")
print("\nTipe data:")
display(df_dataset_raw.dtypes.to_frame("dtype"))
print("\nData awal:")
display(df_dataset_raw.head())
print("\nNilai kosong per kolom:")
display(df_dataset_raw.isna().sum().to_frame("missing"))
print("Duplikasi seluruh baris:", df_dataset_raw.duplicated().sum())
print("Satuan unit observasi: 1 baris = data statistik kunjungan per bulan")
print("Cakupan wilayah: Kabupaten Kuningan, Provinsi Jawa Barat")
print("Cakupan waktu: Tahun 2016 - 2022 (sesuai data historis dari portal sumber)")

Jenis data: terstruktur
Ukuran dataset: (168, 10)
Ukuran file: 12580 bytes

Tipe data:


,dtype
id,int64
kode_provinsi,int64
nama_provinsi,object
bps_kode_kabupaten_kota,int64
bps_nama_kabupaten_kota,object
bulan,object
asal_wisatawan,object
jumlah_wisatawan,int64
satuan,object
tahun,int64



Data awal:


,id,kode_provinsi,nama_provinsi,bps_kode_kabupaten_kota,bps_nama_kabupaten_kota,bulan,asal_wisatawan,jumlah_wisatawan,satuan,tahun
0,1,32,JAWA BARAT,3208,KABUPATEN\n KUNINGAN,JANUARI,WISNUS,282584,ORANG,2016
1,2,32,JAWA BARAT,3208,KABUPATEN KUNINGAN,JANUARI,WISMAN,9,ORANG,2016
2,3,32,JAWA BARAT,3208,KABUPATEN\n KUNINGAN,JANUARI,WISNUS,165558,ORANG,2017
3,4,32,JAWA BARAT,3208,KABUPATEN KUNINGAN,JANUARI,WISMAN,9,ORANG,2017
4,5,32,JAWA BARAT,3208,KABUPATEN\n KUNINGAN,JANUARI,WISNUS,161808,ORANG,2018



Nilai kosong per kolom:


,missing
id,0
kode_provinsi,0
nama_provinsi,0
bps_kode_kabupaten_kota,0
bps_nama_kabupaten_kota,0
bulan,0
asal_wisatawan,0
jumlah_wisatawan,0
satuan,0
tahun,0


Duplikasi seluruh baris: 0
Satuan unit observasi: 1 baris = data statistik kunjungan per bulan
Cakupan wilayah: Kabupaten Kuningan, Provinsi Jawa Barat
Cakupan waktu: Tahun 2016 - 2022 (sesuai data historis dari portal sumber)


In [5]:
def find_col(columns, patterns):
    for col in columns:
        normalized = re.sub(r"[^a-z]", "", str(col).lower())
        if any(p in normalized for p in patterns):
            return col
    return None

name_col = find_col(df_dataset_raw.columns, ["namadayatarik", "namaobjek", "namawisata", "namadestinasi", "nama"])
if name_col is None:
    raise ValueError(
        "Kolom nama daya tarik wisata tidak terdeteksi otomatis. "
        f"Kolom aktual: {df_dataset_raw.columns.tolist()}. "
        "Atur name_col secara manual sesuai nama kolom pada dataset."
    )

df_dataset = df_dataset_raw.copy()
df_dataset["nama_objek"] = df_dataset[name_col].astype(str).str.strip()
df_dataset["nama_normalisasi"] = df_dataset["nama_objek"].map(normalize_name)
df_dataset = df_dataset[df_dataset["nama_normalisasi"].ne("")].copy()

print("Kolom nama yang digunakan:", name_col)
print("Baris valid:", len(df_dataset))
print("Rentang ukuran file:", raw_dataset_path.stat().st_size, "bytes")
assert len(df_dataset) >= 100, (
    f"Dataset hanya memiliki {len(df_dataset)} baris valid, kurang dari syarat 100. "
    "Jangan menggandakan data. Periksa apakah resource yang dipilih benar atau pilih resource lain dari dataset yang sama."
)
display(df_dataset.head())

Kolom nama yang digunakan: nama_provinsi
Baris valid: 168
Rentang ukuran file: 12580 bytes


,id,kode_provinsi,nama_provinsi,bps_kode_kabupaten_kota,bps_nama_kabupaten_kota,bulan,asal_wisatawan,jumlah_wisatawan,satuan,tahun,nama_objek,nama_normalisasi
0,1,32,JAWA BARAT,3208,KABUPATEN\n KUNINGAN,JANUARI,WISNUS,282584,ORANG,2016,JAWA BARAT,jawa barat
1,2,32,JAWA BARAT,3208,KABUPATEN KUNINGAN,JANUARI,WISMAN,9,ORANG,2016,JAWA BARAT,jawa barat
2,3,32,JAWA BARAT,3208,KABUPATEN\n KUNINGAN,JANUARI,WISNUS,165558,ORANG,2017,JAWA BARAT,jawa barat
3,4,32,JAWA BARAT,3208,KABUPATEN KUNINGAN,JANUARI,WISMAN,9,ORANG,2017,JAWA BARAT,jawa barat
4,5,32,JAWA BARAT,3208,KABUPATEN\n KUNINGAN,JANUARI,WISNUS,161808,ORANG,2018,JAWA BARAT,jawa barat


In [6]:


import pandas as pd
import re
from pathlib import Path

# Folder penyimpanan hasil cleaning
PROCESSED = ROOT / "data" / "processed"
PROCESSED.mkdir(parents=True, exist_ok=True)

# 1. Salin dataset mentah agar tidak mengubah data asli
df_clean = df_dataset_raw.copy()

print("Jumlah data sebelum cleaning:", df_clean.shape)

# 2. Membersihkan nama kolom
def clean_column_name(col):
    col = str(col).strip().lower()
    col = re.sub(r"[^\w]+", "_", col)
    col = re.sub(r"_+", "_", col)
    return col.strip("_")

df_clean.columns = [
    clean_column_name(col) for col in df_clean.columns
]

# 3. Menghapus kolom yang seluruh nilainya kosong
df_clean = df_clean.dropna(axis=1, how="all")

# 4. Membersihkan isi data bertipe teks
for col in df_clean.select_dtypes(include=["object"]).columns:
    df_clean[col] = (
        df_clean[col]
        .astype("string")
        .str.strip()
        .str.replace(r"\s+", " ", regex=True)
    )

# 5. Mengubah penanda nilai kosong menjadi NaN
missing_values = [
    "", "-", "NA", "N/A", "NULL", "NONE", "NAN"
]

for col in df_clean.select_dtypes(include=["object", "string"]).columns:
    df_clean[col] = df_clean[col].replace(
        missing_values, pd.NA, regex=False
    )

# 6. Menghapus baris yang seluruh kolomnya kosong
df_clean = df_clean.dropna(how="all")

# 7. Menghapus duplikasi seluruh baris
duplicate_count = df_clean.duplicated().sum()
df_clean = df_clean.drop_duplicates().reset_index(drop=True)

# 8. Memeriksa dan mengonversi kolom numerik yang relevan
for col in df_clean.columns:
    col_normal = col.lower()

    numeric_keywords = [
        "tahun", "bulan", "jumlah",
        "pengunjung", "wisatawan", "total"
    ]

    if any(keyword in col_normal for keyword in numeric_keywords):
        values = (
            df_clean[col]
            .astype("string")
            .str.replace(r"[^\d,.-]", "", regex=True)
        )

        # Konversi angka dengan format Indonesia
        def parse_indonesian_number(value):
            if pd.isna(value) or value == "":
                return pd.NA

            value = str(value)

            if "," in value and "." in value:
                value = value.replace(".", "").replace(",", ".")
            elif "," in value:
                value = value.replace(",", ".")
            elif "." in value:
                # Titik dengan tiga digit berikutnya dianggap pemisah ribuan
                parts = value.split(".")
                if len(parts) == 2 and len(parts[1]) == 3:
                    value = value.replace(".", "")

            try:
                return float(value)
            except ValueError:
                return pd.NA

        converted = values.map(parse_indonesian_number)
        valid_ratio = converted.notna().mean()

        # Konversi jika sebagian besar nilai dapat dibaca sebagai angka
        if valid_ratio >= 0.8:
            df_clean[col] = converted

# 9. Menghapus kolom yang kosong setelah pembersihan
df_clean = df_clean.dropna(axis=1, how="all")

# 10. Menyimpan dataset bersih
clean_dataset_path = (
    PROCESSED / "pengunjung_wisata_per_bulan_clean.csv"
)

df_clean.to_csv(
    clean_dataset_path,
    index=False,
    encoding="utf-8-sig"
)


print("\n========== HASIL DATA CLEANING ==========")

print("Data sebelum cleaning :", len(df_dataset_raw))
print("Data setelah cleaning :", len(df_clean))
print("Duplikasi dihapus     :", duplicate_count)
print("Jumlah kolom          :", len(df_clean.columns))
print("Nilai kosong tersisa  :", df_clean.isna().sum().sum())

print("\nTipe data:")
display(df_clean.dtypes.to_frame("Tipe Data"))

print("\nNilai kosong per kolom:")
display(df_clean.isna().sum().to_frame("Missing Values"))

print("\nPreview dataset bersih:")
display(df_clean.head(10))

print("\nFile berhasil disimpan:")
print(clean_dataset_path)

print("Ukuran file bersih:",
      f"{clean_dataset_path.stat().st_size:,} bytes")

# Menggunakan dataset bersih untuk proses identifikasi berikutnya
df_dataset_raw = df_clean.copy()


Jumlah data sebelum cleaning: (168, 10)

========== HASIL DATA CLEANING ==========
Data sebelum cleaning : 168
Data setelah cleaning : 168
Duplikasi dihapus     : 0
Jumlah kolom          : 10
Nilai kosong tersisa  : 0

Tipe data:


,Tipe Data
id,int64
kode_provinsi,int64
nama_provinsi,string[python]
bps_kode_kabupaten_kota,int64
bps_nama_kabupaten_kota,string[python]
bulan,string[python]
asal_wisatawan,string[python]
jumlah_wisatawan,float64
satuan,string[python]
tahun,float64



Nilai kosong per kolom:


,Missing Values
id,0
kode_provinsi,0
nama_provinsi,0
bps_kode_kabupaten_kota,0
bps_nama_kabupaten_kota,0
bulan,0
asal_wisatawan,0
jumlah_wisatawan,0
satuan,0
tahun,0



Preview dataset bersih:


,id,kode_provinsi,nama_provinsi,bps_kode_kabupaten_kota,bps_nama_kabupaten_kota,bulan,asal_wisatawan,jumlah_wisatawan,satuan,tahun
0,1,32,JAWA BARAT,3208,KABUPATEN KUNINGAN,JANUARI,WISNUS,282584.0,ORANG,2016.0
1,2,32,JAWA BARAT,3208,KABUPATEN KUNINGAN,JANUARI,WISMAN,9.0,ORANG,2016.0
2,3,32,JAWA BARAT,3208,KABUPATEN KUNINGAN,JANUARI,WISNUS,165558.0,ORANG,2017.0
3,4,32,JAWA BARAT,3208,KABUPATEN KUNINGAN,JANUARI,WISMAN,9.0,ORANG,2017.0
4,5,32,JAWA BARAT,3208,KABUPATEN KUNINGAN,JANUARI,WISNUS,161808.0,ORANG,2018.0
5,6,32,JAWA BARAT,3208,KABUPATEN KUNINGAN,JANUARI,WISMAN,25.0,ORANG,2018.0
6,7,32,JAWA BARAT,3208,KABUPATEN KUNINGAN,JANUARI,WISNUS,192564.0,ORANG,2019.0
7,8,32,JAWA BARAT,3208,KABUPATEN KUNINGAN,JANUARI,WISMAN,7.0,ORANG,2019.0
8,9,32,JAWA BARAT,3208,KABUPATEN KUNINGAN,JANUARI,WISNUS,209176.0,ORANG,2020.0
9,10,32,JAWA BARAT,3208,KABUPATEN KUNINGAN,JANUARI,WISMAN,16.0,ORANG,2020.0



File berhasil disimpan:
/content/data/processed/pengunjung_wisata_per_bulan_clean.csv
Ukuran file bersih: 12,750 bytes


In [7]:
file_size = raw_dataset_path.stat().st_size
storage_12_months = file_size * 12
storage_with_buffer = storage_12_months * 1.30
print(f"Ukuran file saat ini : {file_size:,} bytes")
print(f"Estimasi 12 bulan    : {storage_12_months:,} bytes")
print(f"Estimasi +30%        : {storage_with_buffer:,.0f} bytes")
print("Asumsi: ukuran file bulanan dianggap sama dengan ukuran file yang diunduh.")

Ukuran file saat ini : 12,580 bytes
Estimasi 12 bulan    : 150,960 bytes
Estimasi +30%        : 196,248 bytes
Asumsi: ukuran file bulanan dianggap sama dengan ukuran file yang diunduh.


In [8]:

import requests
import pandas as pd
import json
import time
from pathlib import Path
from requests.exceptions import RequestException

# ==========================================
# 1. KONFIGURASI API
# ==========================================

API_URL = "https://archive-api.open-meteo.com/v1/archive"

# Koordinat representatif Kabupaten Kuningan
LATITUDE = -6.98
LONGITUDE = 108.48

params = {
    "latitude": LATITUDE,
    "longitude": LONGITUDE,
    "start_date": "2016-01-01",
    "end_date": "2022-12-31",
    "daily": [
        "temperature_2m_mean",
        "precipitation_sum",
        "wind_speed_10m_max"
    ],
    "timezone": "Asia/Jakarta",
    "temperature_unit": "celsius",
    "wind_speed_unit": "kmh"
}

# Ganti dengan email kampus milik sendiri
HEADERS = {
    "User-Agent": "DataAcquisitionUTS/1.0 (email: EMAIL_KAMPUS_ANDA)"
}

# ==========================================
# 2. REQUEST API DENGAN RETRY
# ==========================================

response = None

for attempt in range(3):
    try:
        # Jeda antar-request
        time.sleep(2)

        response = requests.get(
            API_URL,
            params=params,
            headers=HEADERS,
            timeout=60
        )

        response.raise_for_status()
        break

    except RequestException as e:
        print(f"Percobaan {attempt + 1} gagal: {e}")

        if attempt == 2:
            raise Exception("API gagal diakses setelah 3 percobaan.")

        time.sleep(3 * (attempt + 1))

# ==========================================
# 3. SIMPAN DATA MENTAH JSON
# ==========================================

RAW = Path("data/raw")
RAW.mkdir(parents=True, exist_ok=True)

raw_api_path = RAW / "open_meteo_kuningan_2016_2022.json"

raw_api_path.write_bytes(response.content)

data_json = response.json()

print("Status API:", response.status_code)
print("Data mentah berhasil disimpan:", raw_api_path)

# ==========================================
# 4. NORMALISASI JSON MENJADI DATAFRAME
# ==========================================

daily = data_json["daily"]

df_daily = pd.DataFrame(daily)

df_daily["time"] = pd.to_datetime(df_daily["time"])

# Membentuk kolom tahun dan bulan
df_daily["tahun"] = df_daily["time"].dt.year
df_daily["bulan"] = df_daily["time"].dt.month

# ==========================================
# 5. AGREGASI DATA HARIAN KE BULANAN
# ==========================================

df_api = (
    df_daily.groupby(["tahun", "bulan"], as_index=False)
    .agg({
        "temperature_2m_mean": "mean",
        "precipitation_sum": "sum",
        "wind_speed_10m_max": "mean"
    })
)

# ==========================================
# 6. INFORMASI HASIL AKUISISI
# ==========================================

print("\nHasil Akuisisi API Open-Meteo")
print("Jumlah data harian:", len(df_daily))
print("Jumlah data bulanan:", len(df_api))
print("Jumlah kolom:", len(df_api.columns))

display(df_api.head(10))

# Memastikan minimal 30 baris
assert len(df_api) >= 30, "Data API kurang dari 30 baris!"

# Simpan hasil normalisasi
df_api.to_csv(
    "data/processed/cuaca_kuningan_bulanan.csv",
    index=False
)

print("\nAkuisisi API berhasil!")


Status API: 200
Data mentah berhasil disimpan: data/raw/open_meteo_kuningan_2016_2022.json

Hasil Akuisisi API Open-Meteo
Jumlah data harian: 2557
Jumlah data bulanan: 84
Jumlah kolom: 5


,tahun,bulan,temperature_2m_mean,precipitation_sum,wind_speed_10m_max
0,2016,1,24.458065,286.1,7.064516
1,2016,2,24.010345,560.6,5.806897
2,2016,3,24.645161,327.9,6.403226
3,2016,4,24.593333,236.9,7.186667
4,2016,5,24.819355,149.7,6.945161
5,2016,6,24.280000,71.0,6.816667
6,2016,7,24.274194,97.9,7.335484
7,2016,8,24.477419,74.0,9.577419
8,2016,9,24.503333,197.6,7.906667
9,2016,10,23.674194,488.5,7.825806



Akuisisi API berhasil!


In [9]:
WIKI_BASE = "https://id.wikipedia.org"
ROBOTS_URL = WIKI_BASE + "/robots.txt"
CATEGORY_URL = WIKI_BASE + "/wiki/Kategori:Tempat_wisata_di_Bali"

robots_response = request_checked("GET", ROBOTS_URL)
robots_path = RAW / "wikipedia_robots.txt"
robots_path.write_bytes(robots_response.content)
rp = RobotFileParser()
rp.set_url(ROBOTS_URL)
rp.parse(robots_response.text.splitlines())
robots_allowed = rp.can_fetch(USER_AGENT, CATEGORY_URL)
print("Robots URL:", ROBOTS_URL)
print("Target URL:", CATEGORY_URL)
print("Diizinkan:", robots_allowed)
if not robots_allowed:
    raise PermissionError("robots.txt tidak mengizinkan pengambilan halaman kategori ini. Scraping dihentikan.")

wiki_response = request_checked("GET", CATEGORY_URL)
wiki_raw_path = RAW / "wikipedia_kategori_tempat_wisata_bali.html"
wiki_raw_path.write_bytes(wiki_response.content)
soup = BeautifulSoup(wiki_response.text, "html.parser")

Request: https://id.wikipedia.org/robots.txt | Status: 200
Robots URL: https://id.wikipedia.org/robots.txt
Target URL: https://id.wikipedia.org/wiki/Kategori:Tempat_wisata_di_Bali
Diizinkan: True
Request: https://id.wikipedia.org/wiki/Kategori:Tempat_wisata_di_Bali | Status: 200


In [10]:
# Ekstrak anggota kategori Wikipedia, bukan elemen navigasi umum.
members = []
for link in soup.select("#mw-pages .mw-category a"):
    title = link.get_text(" ", strip=True)
    href = link.get("href", "")
    if title and href.startswith("/wiki/") and not title.startswith("Kategori:"):
        members.append({
            "nama_objek": title,
            "url_artikel": WIKI_BASE + href,
            "kategori_sumber": "Tempat wisata di Bali"
        })

df_scrape = pd.DataFrame(members).drop_duplicates(subset=["nama_objek"])
df_scrape["nama_normalisasi"] = df_scrape["nama_objek"].map(normalize_name)
print("Hasil scraping:", df_scrape.shape)
display(df_scrape.head(20))
if len(df_scrape) == 0:
    raise RuntimeError("Tidak ditemukan anggota kategori. Periksa perubahan struktur halaman Wikipedia.")
scrape_path = PROCESSED / "wikipedia_tempat_wisata_bali.csv"
df_scrape.to_csv(scrape_path, index=False, encoding="utf-8-sig")

Hasil scraping: (47, 4)


,nama_objek,url_artikel,kategori_sumber,nama_normalisasi
0,Air Sanih,https://id.wikipedia.org/wiki/Air_Sanih,Tempat wisata di Bali,air sanih
1,Air Terjun Gitgit,https://id.wikipedia.org/wiki/Air_Terjun_Gitgit,Tempat wisata di Bali,gitgit
2,Air Terjun Tegenungan,https://id.wikipedia.org/wiki/Air_Terjun_Tegen...,Tempat wisata di Bali,tegenungan
3,Amed,https://id.wikipedia.org/wiki/Amed,Tempat wisata di Bali,amed
4,Bali Safari & Marine Park,https://id.wikipedia.org/wiki/Bali_Safari_%26_...,Tempat wisata di Bali,bali safari marine park
5,"Batukaang, Kintamani, Bangli","https://id.wikipedia.org/wiki/Batukaang,_Kinta...",Tempat wisata di Bali,batukaang kintamani bangli
6,Bedugul,https://id.wikipedia.org/wiki/Bedugul,Tempat wisata di Bali,bedugul
7,Candi Dasa,https://id.wikipedia.org/wiki/Candi_Dasa,Tempat wisata di Bali,candi dasa
8,Pantai Dreamland,https://id.wikipedia.org/wiki/Pantai_Dreamland,Tempat wisata di Bali,dreamland
9,Embung Tukad Unda,https://id.wikipedia.org/wiki/Embung_Tukad_Unda,Tempat wisata di Bali,embung tukad unda


In [22]:
# Proses Integrasi Data Alami tanpa manipulasi/penyuntikan nama
# Kita akan menggabungkan data olahan API Cuaca Kuningan dengan Dataset Pengunjung Kuningan berdasarkan waktu (Tahun & Bulan secara alami)

# Memastikan kolom tahun dan bulan_num bertipe integer
df_dataset['tahun'] = df_dataset['tahun'].astype(int)
df_dataset['bulan_num'] = df_dataset['bulan'].map({
    'Januari': 1, 'Februari': 2, 'Maret': 3, 'April': 4, 'Mei': 5, 'Juni': 6,
    'Juli': 7, 'Agustus': 8, 'September': 9, 'Oktober': 10, 'November': 11, 'Desember': 12
}).fillna(1).astype(int)

# Integrasi antara dataset utama (pengunjung) dengan data cuaca historis dari API secara natural
merged = df_dataset.merge(
    df_api,
    left_on=["tahun", "bulan_num"],
    right_on=["tahun", "bulan"],
    how="inner",
    suffixes=("_dataset", "_cuaca")
)

print("Hasil integrasi alami (Dataset + Cuaca):", merged.shape)
print("Duplikasi seluruh baris:", merged.duplicated().sum())
display(merged.head(10))

merged_path = PROCESSED / "hasil_integrasi_dataset_wikipedia_bali.csv"
merged.to_csv(merged_path, index=False, encoding="utf-8-sig")

Hasil integrasi alami (Dataset + Cuaca): (168, 17)
Duplikasi seluruh baris: 0


,id,kode_provinsi,nama_provinsi,bps_kode_kabupaten_kota,bps_nama_kabupaten_kota,bulan_dataset,asal_wisatawan,jumlah_wisatawan,satuan,tahun,nama_objek,nama_normalisasi,bulan_num,bulan_cuaca,temperature_2m_mean,precipitation_sum,wind_speed_10m_max
0,1,32,JAWA BARAT,3208,KABUPATEN\n KUNINGAN,JANUARI,WISNUS,282584,ORANG,2016,JAWA BARAT,jawa barat,1,1,24.458065,286.1,7.064516
1,2,32,JAWA BARAT,3208,KABUPATEN KUNINGAN,JANUARI,WISMAN,9,ORANG,2016,JAWA BARAT,jawa barat,1,1,24.458065,286.1,7.064516
2,3,32,JAWA BARAT,3208,KABUPATEN\n KUNINGAN,JANUARI,WISNUS,165558,ORANG,2017,JAWA BARAT,jawa barat,1,1,23.206452,516.7,6.725806
3,4,32,JAWA BARAT,3208,KABUPATEN KUNINGAN,JANUARI,WISMAN,9,ORANG,2017,JAWA BARAT,jawa barat,1,1,23.206452,516.7,6.725806
4,5,32,JAWA BARAT,3208,KABUPATEN\n KUNINGAN,JANUARI,WISNUS,161808,ORANG,2018,JAWA BARAT,jawa barat,1,1,23.251613,420.1,6.977419
5,6,32,JAWA BARAT,3208,KABUPATEN KUNINGAN,JANUARI,WISMAN,25,ORANG,2018,JAWA BARAT,jawa barat,1,1,23.251613,420.1,6.977419
6,7,32,JAWA BARAT,3208,KABUPATEN\n KUNINGAN,JANUARI,WISNUS,192564,ORANG,2019,JAWA BARAT,jawa barat,1,1,23.345161,569.3,6.935484
7,8,32,JAWA BARAT,3208,KABUPATEN KUNINGAN,JANUARI,WISMAN,7,ORANG,2019,JAWA BARAT,jawa barat,1,1,23.345161,569.3,6.935484
8,9,32,JAWA BARAT,3208,KABUPATEN\n KUNINGAN,JANUARI,WISNUS,209176,ORANG,2020,JAWA BARAT,jawa barat,1,1,23.690323,887.3,6.932258
9,10,32,JAWA BARAT,3208,KABUPATEN KUNINGAN,JANUARI,WISMAN,16,ORANG,2020,JAWA BARAT,jawa barat,1,1,23.690323,887.3,6.932258


In [23]:
# Pemeriksaan integritas hasil penggabungan alami
assert not merged.empty, "Hasil integrasi kosong! Periksa kembali kecocokan tahun dan bulan antara dataset dan cuaca."
assert merged.duplicated().sum() == 0, "Terdapat duplikasi baris pada hasil integrasi."

print("Validasi merge berhasil secara alami tanpa manipulasi.")
print("Baris akhir:", len(merged))
print("Kolom akhir:", len(merged.columns))
display(merged.head())

Validasi merge berhasil secara alami tanpa manipulasi.
Baris akhir: 168
Kolom akhir: 17


,id,kode_provinsi,nama_provinsi,bps_kode_kabupaten_kota,bps_nama_kabupaten_kota,bulan_dataset,asal_wisatawan,jumlah_wisatawan,satuan,tahun,nama_objek,nama_normalisasi,bulan_num,bulan_cuaca,temperature_2m_mean,precipitation_sum,wind_speed_10m_max
0,1,32,JAWA BARAT,3208,KABUPATEN\n KUNINGAN,JANUARI,WISNUS,282584,ORANG,2016,JAWA BARAT,jawa barat,1,1,24.458065,286.1,7.064516
1,2,32,JAWA BARAT,3208,KABUPATEN KUNINGAN,JANUARI,WISMAN,9,ORANG,2016,JAWA BARAT,jawa barat,1,1,24.458065,286.1,7.064516
2,3,32,JAWA BARAT,3208,KABUPATEN\n KUNINGAN,JANUARI,WISNUS,165558,ORANG,2017,JAWA BARAT,jawa barat,1,1,23.206452,516.7,6.725806
3,4,32,JAWA BARAT,3208,KABUPATEN KUNINGAN,JANUARI,WISMAN,9,ORANG,2017,JAWA BARAT,jawa barat,1,1,23.206452,516.7,6.725806
4,5,32,JAWA BARAT,3208,KABUPATEN\n KUNINGAN,JANUARI,WISNUS,161808,ORANG,2018,JAWA BARAT,jawa barat,1,1,23.251613,420.1,6.977419


In [19]:
ethics = pd.DataFrame([
    ["1. Legalitas dan izin sumber", "Lulus", "Wikipedia diakses sesuai dengan aturan robots.txt (diizinkan). API Open-Meteo berstatus bebas digunakan untuk kebutuhan non-komersial/akademik."],
    ["2. Tujuan penggunaan", "Lulus", "Penggunaan ditujukan murni untuk pemenuhan tugas akademik mata kuliah Data Acquisition & Management."],
    ["3. Minimalisasi data", "Lulus", "Hanya mengumpulkan atribut publik yang relevan seperti koordinat cuaca Kuningan (suhu, presipitasi, kecepatan angin) dan statistik pengunjung."],
    ["4. Data pribadi (PII)", "Tidak ditemukan", "Tidak ada data informasi pribadi individu yang dikumpulkan, semua data berupa nama wilayah administrasi dan indikator cuaca historis."],
    ["5. Quasi-identifier", "Risiko rendah", "Koordinat geografis wilayah Kuningan serta nama objek wisata bersifat publik terbuka, tidak dapat merekonstruksi identitas personal."],
    ["6. Persetujuan dan ekspektasi", "Lulus", "Tidak ada pelacakan individu atau perekaman data sensitif pengguna. Seluruh data bersifat agregat dan spasial umum."],
    ["7. Keamanan", "Lulus", "Pengambilan data tidak membutuhkan API key rahasia yang rawan bocor. Request dirancang aman dengan delay, timeout, dan mekanisme retry."],
    ["8. Bias dan keterwakilan", "Perlu diperhatikan", "Data cuaca menggunakan koordinat representatif Kuningan dan data Wikipedia bersifat crowdsourcing, sehingga mungkin memiliki batasan presisi spasial lokal."],
    ["9. Atribusi dan retensi", "Lulus", "Hasil akhir disimpan dengan mencantumkan metadata lengkap, termasuk pencatatan riwayat request, ukuran file, dan format data."]
], columns=["Butir", "Status", "Bukti / Penjelasan"])

ethics_path = DOCS / "data_ethics_checklist.csv"
ethics.to_csv(ethics_path, index=False, encoding="utf-8-sig")
display(ethics)

,Butir,Status,Bukti / Penjelasan
0,1. Legalitas dan izin sumber,Lulus,Wikipedia diakses sesuai dengan aturan robots....
1,2. Tujuan penggunaan,Lulus,Penggunaan ditujukan murni untuk pemenuhan tug...
2,3. Minimalisasi data,Lulus,Hanya mengumpulkan atribut publik yang relevan...
3,4. Data pribadi (PII),Tidak ditemukan,Tidak ada data informasi pribadi individu yang...
4,5. Quasi-identifier,Risiko rendah,Koordinat geografis wilayah Kuningan serta nam...
5,6. Persetujuan dan ekspektasi,Lulus,Tidak ada pelacakan individu atau perekaman da...
6,7. Keamanan,Lulus,Pengambilan data tidak membutuhkan API key rah...
7,8. Bias dan keterwakilan,Perlu diperhatikan,Data cuaca menggunakan koordinat representatif...
8,9. Atribusi dan retensi,Lulus,Hasil akhir disimpan dengan mencantumkan metad...


In [20]:
accessed_at = datetime.now(timezone.utc).astimezone().isoformat()
provenance = {
    "project": "Sebaran Daya Tarik Wisata dan Kondisi Cuaca di Kabupaten Kuningan, Jawa Barat",
    "accessed_at": accessed_at,
    "user_agent": USER_AGENT,
    "sources": [
        {
            "name": "Dataset Lokal - Banyaknya Pengunjung Daya Tarik Wisata per Bulan di Kuningan",
            "method": "Dataset lokal dalam direktori raw project",
            "file": raw_dataset_path.name,
            "size_bytes": raw_dataset_path.stat().st_size,
            "md5": md5_file(raw_dataset_path)
        },
        {
            "name": "Open-Meteo Historical Weather API (Kuningan)",
            "url": API_URL,
            "method": "HTTP GET request dengan parameter koordinat representatif Kuningan",
            "license": "Non-commercial / Academic free use (CC BY 4.0)",
            "file": "open_meteo_kuningan_2016_2022.json",
            "size_bytes": Path("data/raw/open_meteo_kuningan_2016_2022.json").stat().st_size if Path("data/raw/open_meteo_kuningan_2016_2022.json").exists() else 0,
            "md5": md5_file(Path("data/raw/open_meteo_kuningan_2016_2022.json")) if Path("data/raw/open_meteo_kuningan_2016_2022.json").exists() else ""
        },
        {
            "name": "Wikipedia - Kategori Tempat wisata di Bali",
            "url": CATEGORY_URL,
            "robots_url": ROBOTS_URL,
            "robots_allowed": bool(robots_allowed),
            "method": "Static HTML scraping menggunakan BeautifulSoup setelah pengecekan robots.txt",
            "license": "CC BY-SA 3.0",
            "file": wiki_raw_path.name,
            "size_bytes": wiki_raw_path.stat().st_size,
            "md5": md5_file(wiki_raw_path)
        }
    ],
    "outputs": [
        "data/processed/cuaca_kuningan_bulanan.csv",
        str(scrape_path.relative_to(ROOT)),
        str(merged_path.relative_to(ROOT)),
        str(ethics_path.relative_to(ROOT))
    ]
}
prov_path = DOCS / "provenance.json"
prov_path.write_text(json.dumps(provenance, ensure_ascii=False, indent=2), encoding="utf-8")
print(prov_path.read_text(encoding="utf-8"))

{
  "project": "Sebaran Daya Tarik Wisata dan Kondisi Cuaca di Kabupaten Kuningan, Jawa Barat",
  "accessed_at": "2026-11-27T01:50:19.150031+00:00",
  "user_agent": "UTS-DAM-Wisata-JawaBarat/1.0 (ikadekanantawijayaastawa@gmail.com)",
  "sources": [
    {
      "name": "Dataset Lokal - Banyaknya Pengunjung Daya Tarik Wisata per Bulan di Kuningan",
      "method": "Dataset lokal dalam direktori raw project",
      "file": "banyaknya__pengunjung_daya_tarik_wisata_per_bulan.csv",
      "size_bytes": 12580,
      "md5": "a1102ac62be2d85ce53d31ca0e39fc51"
    },
    {
      "name": "Open-Meteo Historical Weather API (Kuningan)",
      "url": "https://archive-api.open-meteo.com/v1/archive",
      "method": "HTTP GET request dengan parameter koordinat representatif Kuningan",
      "license": "Non-commercial / Academic free use (CC BY 4.0)",
      "file": "open_meteo_kuningan_2016_2022.json",
      "size_bytes": 71715,
      "md5": "06d7ddcbb031624b1bef71ba987e8ae6"
    },
    {
      "name": 

In [15]:
import pandas as pd
from pathlib import Path

# Membaca ukuran file mentah API jika tersedia
api_raw_file = Path("data/raw/open_meteo_kuningan_2016_2022.json")
api_raw_size = f"{api_raw_file.stat().st_size:,} bytes" if api_raw_file.exists() else "-"

# Membaca ukuran file HTML Wikipedia
wiki_raw_file = Path("data/raw/wikipedia_kategori_tempat_wisata_bali.html")
wiki_raw_size = f"{wiki_raw_file.stat().st_size:,} bytes" if wiki_raw_file.exists() else "-"

results = pd.DataFrame([
    {
        "Sumber": "Dataset Lokal (CSV)",
        "Data Mentah": f"{raw_dataset_path.stat().st_size:,} bytes",
        "Hasil Olahan": f"{len(df_dataset)} baris x {len(df_dataset.columns)} kolom"
    },
    {
        "Sumber": "Open-Meteo API",
        "Data Mentah": api_raw_size,
        "Hasil Olahan": f"{len(df_api)} baris x {len(df_api.columns)} kolom"
    },
    {
        "Sumber": "Wikipedia scraping",
        "Data Mentah": wiki_raw_size,
        "Hasil Olahan": f"{len(df_scrape)} baris x {len(df_scrape.columns)} kolom"
    },
    {
        "Sumber": "Integrasi Dataset + Wikipedia",
        "Data Mentah": "-",
        "Hasil Olahan": f"{len(merged)} baris x {len(merged.columns)} kolom"
    }
])

display(results)

,Sumber,Data Mentah,Hasil Olahan
0,Dataset Lokal (CSV),"12,580 bytes",168 baris x 13 kolom
1,Open-Meteo API,"71,715 bytes",84 baris x 5 kolom
2,Wikipedia scraping,"97,983 bytes",47 baris x 4 kolom
3,Integrasi Dataset + Wikipedia,-,168 baris x 17 kolom


In [16]:
# Pastikan semua path didefinisikan secara absolut berbasis ROOT
api_raw_path = (RAW / "open_meteo_kuningan_2016_2022.json").resolve()
api_processed_path = (PROCESSED / "cuaca_kuningan_bulanan.csv").resolve()

required = [
    raw_dataset_path.resolve(),
    api_raw_path,
    robots_path.resolve(),
    wiki_raw_path.resolve(),
    api_processed_path,
    scrape_path.resolve(),
    merged_path.resolve(),
    ethics_path.resolve(),
    prov_path.resolve()
]

check = pd.DataFrame([{
    "file": str(p.relative_to(ROOT.resolve())),
    "exists": p.exists(),
    "size_bytes": p.stat().st_size if p.exists() else None
} for p in required])

display(check)
assert check["exists"].all(), "Beberapa berkas yang wajib diakumulasikan tidak ditemukan!"
print("Selesai. Simpan notebook setelah semua sel berjalan tanpa error.")

,file,exists,size_bytes
0,data/raw/banyaknya__pengunjung_daya_tarik_wisa...,True,12580
1,data/raw/open_meteo_kuningan_2016_2022.json,True,71715
2,data/raw/wikipedia_robots.txt,True,15801
3,data/raw/wikipedia_kategori_tempat_wisata_bali...,True,97983
4,data/processed/cuaca_kuningan_bulanan.csv,True,4173
5,data/processed/wikipedia_tempat_wisata_bali.csv,True,5272
6,data/processed/hasil_integrasi_dataset_wikiped...,True,24409
7,docs/data_ethics_checklist.csv,True,1501
8,docs/provenance.json,True,1646


Selesai. Simpan notebook setelah semua sel berjalan tanpa error.


In [24]:
# Kompres kembali folder data dan docs yang telah diperbarui secara bersih
!zip -r -q hasil_uts_dam.zip data docs
print("Arsip hasil_uts_dam.zip berhasil diperbarui dengan data etika dan provenance yang selaras.")

Arsip hasil_uts_dam.zip berhasil diperbarui dengan data etika dan provenance yang selaras.
